# Fair live semantic HNSW — reproducibility notebook

This notebook is a thin Colab wrapper around `experiments.semantic_hnsw_live_sweep`. It does **not** patch source code at runtime. The Rust executable already uses the normalized-dot distance used by both the library HNSW baseline and the custom traversal.

The benchmark exports real held-out fashion assets, derives semantic selectivity thresholds, executes real Binary1-LS2-int4 programs inside timed HNSW traversal, and writes raw + summary + environment artifacts.


In [ ]:
#@title 1) Settings
FULL_DATA = True #@param {type:"boolean"}
QUERIES = 100 #@param {type:"integer"}
K = 50 #@param {type:"integer"}
EF = 128 #@param {type:"integer"}
POSITIVE = 'minimalist,office_appropriate' #@param {type:"string"}
NEGATIVE = 'technical_sporty' #@param {type:"string"}
FRACTIONS = '0.50,0.20,0.10,0.05,0.02' #@param {type:"string"}


In [ ]:
#@title 2) Clone repository and install dependencies
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[benchmark]'], check=True)
if shutil.which('rustc') is None or shutil.which('cargo') is None:
    subprocess.run(['bash','-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
    os.environ['PATH'] = str(pathlib.Path.home()/'.cargo'/'bin') + os.pathsep + os.environ.get('PATH','')
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('python:', sys.version.split()[0])
print('rustc:', subprocess.check_output(['rustc','--version'], text=True).strip())


In [ ]:
#@title 3) Run the complete benchmark
import subprocess, sys, pathlib, os
os.chdir('/content/ras')
CFG = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
OUT = pathlib.Path('/content/semantic_hnsw_live_fair')
cmd = [sys.executable, '-m', 'experiments.semantic_hnsw_live_sweep',
       '--config', CFG, '--output-dir', str(OUT),
       '--positive', POSITIVE, '--negative', NEGATIVE,
       '--fractions', FRACTIONS, '--queries', str(QUERIES),
       '--k', str(K), '--ef', str(EF)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 4) Inspect paper-grade outputs
import pandas as pd, json, pathlib
OUT = pathlib.Path('/content/semantic_hnsw_live_fair')
fair = pd.read_csv(OUT/'fairness.csv')
summary = pd.read_csv(OUT/'summary.csv')
display(fair)
display(summary.sort_values(['target_fraction','mean_latency_ms'], ascending=[False, True]))
print(json.loads((OUT/'environment.json').read_text()))


In [ ]:
#@title 5) Package artifacts
import shutil
shutil.make_archive('/content/semantic_hnsw_live_fair', 'zip', '/content/semantic_hnsw_live_fair')
print('/content/semantic_hnsw_live_fair.zip')
